In [8]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- SETTINGS ---
input_folder = "../../../FIRST-Robotics-Competition-Data-Challenge-Videos/uncropped_videos"      # Folder containing your .mp4s
output_folder = "../../../FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos"       # Where the new files will go

def find_split_point(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        return None

    # 1. Convert to grayscale and blur slightly to reduce noise
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # 2. Sum the brightness along the horizontal axis (rows)
    # This creates a 1D array of the average intensity of each row
    row_sums = np.mean(gray, axis=1)

    # 3. We look for the minimum intensity in the middle 20% of the frame
    # (Assuming the split isn't at the very edge)
    mid = len(row_sums) // 2
    search_range = slice(mid - 100, mid + 100) 
    split_point = np.argmin(row_sums[search_range]) + (mid - 100)

    return split_point, row_sums

def crop_and_save_video(input_path, output_path, split_x):
    # 1. Open the source video
    cap = cv2.VideoCapture(input_path)
    
    # Get original video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    orig_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # 2. Define the NEW dimensions (Cropping to the left POV)
    new_height = int(orig_height - split_y)
    new_width = orig_width
    
    # 3. Define the Codec and create VideoWriter object
    # 'mp4v' is a standard codec for .mp4 files
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (new_width, new_height))
    
    print(f"Processing... Exporting to {output_path}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # 4. Perform the Crop [y_start:y_end, x_start:x_end]
        # To get the left POV: [0:height, 0:split_x]
        # To get the right POV: [0:height, split_x:orig_width]
        cropped_frame = frame[split_y:orig_height, 0:new_width]
        
        # 5. Write the cropped frame to the new file
        out.write(cropped_frame)
        
    # Release everything when finished
    cap.release()
    out.release()
    print("Done! Video saved successfully.")

In [9]:
# Usage
video_file = "f1m1.mp4"
input_file = input_folder + "/" + video_file
output_file = output_folder + "/cropped_" + video_file
split_y, intensity_profile = find_split_point(input_file)

print(f"Detected Split Point at Y-coordinate: {split_y}")

# Example Usage:
crop_and_save_video(input_file, output_file, split_y)

Detected Split Point at Y-coordinate: 452
Processing... Exporting to ../../../FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_f1m1.mp4
Done! Video saved successfully.
